# rift `nisar2cog` — DPS job runner (bounding-box driven)

Query ASF for NISAR **GSLC** granules over an area of interest (provisional PR products,
40 MHz freq-A via `rangeBandwidth="40+5"`), and submit one MAAP DPS job per granule to the
`rift-nisar2cog` OGC process. Start on the **`maap-dps-sandbox`** queue.

Run this in a **MAAP Hub (OGC) workspace** so maap-py v5 is available.

**Prereqs**
- The `rift-nisar2cog` OGC process is deployed (via `.github/workflows/ogc-app-pack.yml`).
- `asf-search` installed (`pip install asf-search`).

In [ ]:
# One-time in a fresh workspace:
# %pip install asf-search
import asf_search as asf
import pandas as pd
import datetime, json, os, time
from maap.maap import MAAP

maap = MAAP()
print("asf-search", asf.__version__)

## 1. Area of interest

Thwaites / Pine Island sector. We pass the WKT polygon straight to `intersectsWith=` for
precise spatial filtering (more accurate than a bbox).

In [ ]:
AOI_WKT = ("POLYGON((-101.9351 -74.7848,-102.4704 -75.4215,"
           "-99.852 -75.5511,-99.4242 -74.9087,-101.9351 -74.7848))")

# Equivalent bounding box (MINX MINY MAXX MAXY), for reference / STAC fallbacks:
AOI_BBOX = "-102.4704 -75.5511 -99.4242 -74.7848"
AOI_WKT

## 2. Search ASF for NISAR GSLC granules

We query ASF with the exact NISAR filters (from the ASF Vertex search builder):
- `dataset=["NISAR"]`, `processingLevel=["GSLC"]`
- `rangeBandwidth=["40+5"]` — **server-side** 40 MHz frequency-A filter (no post-filtering needed)
- `dataMaturity=["PROVISIONAL"]`, `productionConfiguration=["PR"]`
- a `start`/`end` acquisition window
- `intersectsWith=<AOI WKT>`

NISAR search results may require Earthdata Login; if `asf.search` returns 0 or errors on
auth, uncomment the session block below and enter EDL credentials (or a CMR token).

In [ ]:
# Acquisition window (adjust as needed).
START = "2026-07-08T07:00:00Z"
END   = "2026-07-20T06:59:59Z"

opts = asf.ASFSearchOptions(
    maxResults=250,
    dataset=["NISAR"],
    processingLevel=["GSLC"],
    rangeBandwidth=["40+5"],          # 40 MHz freq-A, server-side
    dataMaturity=["PROVISIONAL"],
    productionConfiguration=["PR"],
    start=START,
    end=END,
    intersectsWith=AOI_WKT,
)

# NISAR results may require Earthdata Login. If len(results)==0 or you get an auth error,
# uncomment and provide credentials (or use session.auth_with_token(getpass('EDL Token'))):
# from getpass import getpass
# session = asf.ASFSession()
# session.auth_with_creds(input("EDL Username: "), getpass("EDL Password: "))
# opts.session = session

results = asf.search(opts=opts)
print(f"matched {len(results)} granules")
len(results)

In [ ]:
# Peek at the first result's properties (sanity check: fileID, url, rangeBandwidth, etc.).
if len(results):
    print(json.dumps(results[0].properties, indent=2, default=str))
else:
    print("No results — widen the date window, check the AOI, or add EDL auth (see cell above).")

## 3. Build the granule URL list

`rangeBandwidth=["40+5"]` already filtered to 40 MHz freq-A server-side, so no client-side
filtering is needed — we just collect each granule's download URL.

**Heads-up on auth:** these ASF URLs typically require Earthdata Login. The DPS worker's
`run.py` downloads `gslc_url` with plain `requests` (no credentials), so a bare ASF URL may
401/403 on the worker. If so, stage the granules into your `my-public-bucket`
(`s3://maap-ops-workspace/shared/<username>/...`) and submit those anonymous-HTTPS URLs
instead. Verify with the single sandbox job before scaling.

In [ ]:
# Collect (granuleName, url) for each result. The .h5 GSLC is the default/primary product URL.
granules = []
for r in results:
    p = r.properties
    url = p.get("url")
    name = p.get("fileID") or p.get("sceneName") or p.get("fileName")
    if url:
        granules.append((name, url))

print(f"{len(granules)} granule URLs")
for name, url in granules[:5]:
    print(" ", name, "->", url)

gslc_urls = [u for _, u in granules]

## 4. Resolve the deployed OGC process id

`maap.list_algorithms()` returns `{"processes": [{"id":..., "title":...}, ...]}`. Find the
`rift-nisar2cog` entry and grab its `id`.

In [ ]:
resp = maap.list_algorithms()
procs = resp.json().get("processes", []) if resp.status_code == 200 else []
for p in procs:
    print(p.get("id"), "|", p.get("title"))

PROCESS_ID = next((p["id"] for p in procs
                   if "nisar2cog" in str(p.get("id", "")).lower()
                   or "nisar2cog" in str(p.get("title", "")).lower()), None)
print("\nPROCESS_ID =", PROCESS_ID)

## 5. Submit one DPS job per granule

Start with a **single** granule (`kept_urls[:1]`) on `maap-dps-sandbox` to validate before
scaling. `submit_job` returns HTTP 202 with `{"id": ..., "status": "accepted"}`.

In [ ]:
QUEUE = "maap-dps-sandbox"        # 8 GB, 10-min cap. Fall back to maap-dps-worker-16gb/-32gb.
TAG = "nisar2cog-thwaites"
POLS = ""                          # empty = all freq-A pols
AMP_ONLY = "false"

TEST_URLS = gslc_urls[:1]          # <-- widen to gslc_urls once the first job succeeds
assert PROCESS_ID, "PROCESS_ID not resolved — is the process deployed?"
assert TEST_URLS, "No granule URLs — check the search cells."

rows = []
for i, url in enumerate(TEST_URLS, start=1):
    inputs = {"gslc_url": url, "pols": POLS, "amp_only": AMP_ONLY}
    r = maap.submit_job(process_id=PROCESS_ID, inputs=inputs,
                        queue=QUEUE, dedup=True, tag=TAG)
    body = r.json() if r.status_code == 202 else {}
    job_id, status = body.get("id"), body.get("status", r.text)
    print(f"[{i}/{len(TEST_URLS)}] {r.status_code} job_id={job_id} status={status}")
    rows.append({"n": i, "gslc_url": url, "job_id": job_id,
                 "submit_status": status, "http": r.status_code,
                 "submit_time": datetime.datetime.now().isoformat()})

submit_df = pd.DataFrame(rows)
out_dir = os.path.expanduser("~/my-public-bucket/dps_submission_results")
os.makedirs(out_dir, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y%m%d%H%M")
csv_path = f"{out_dir}/nisar2cog_{TAG}_{stamp}.csv"
submit_df.to_csv(csv_path, index=False)
print("saved", csv_path)
submit_df

## 6. Monitor jobs and fetch results

`get_job_status` / `get_job_result` return JSON in maap-py v5. Results land under
`~/my-private-bucket/dps_output/rift-nisar2cog/...`.

In [ ]:
for job_id in [j for j in submit_df["job_id"].tolist() if j]:
    s = maap.get_job_status(job_id)
    st = s.json().get("status") if s.status_code == 200 else s.text
    print(job_id, "->", st)

In [ ]:
# Once a job shows succeeded, inspect its outputs:
SUCCESS_JOB_ID = ""  # paste a job id
if SUCCESS_JOB_ID:
    r = maap.get_job_result(SUCCESS_JOB_ID)
    print(json.dumps(r.json(), indent=2) if r.status_code == 200 else r.text)
    # metrics (OGC v5 only):
    m = maap.get_job_metrics(SUCCESS_JOB_ID)
    if m.status_code == 200:
        print(json.dumps(m.json(), indent=2))